# Decision-payoff selection vs. statistical accuracy

A simple binary-gamble example to make one point: **agents at different risk-aversion levels pick the model whose biases align with their own decision-loss profile, not the model that is closest to the truth.** Three agents with different $\gamma$ can rationally select different models from the same set of candidates, even on the same data and the same procedure.

**The true gamble.** Each period the payoff multiple is $M_+ = 1.5$ with true probability $p = 0.55$, otherwise $M_- = 0.7$. Per-period wealth evolves as
$$
W_{t+1} = W_t \cdot \bigl[1 + f(M - 1)\bigr], \qquad M \in \{M_+, M_-\}.
$$
The true mean payoff is $0.55 \cdot 1.5 + 0.45 \cdot 0.7 = 1.14$.

**The agents have three candidate models, previously fit in a training set.** All three are misspecified, none matches the truth. But by construction **all three agree with the truth on the mean payoff (1.14)**; they only disagree on how severe the left tail is — i.e., on how bad the loss state $M_-$ is.

| Model | $q$ | $M_+$ | $M_-$ | character |
|---|---:|---:|---:|---|
| A — thin left tail | 0.31 | 1.897 | 0.80 | rare big wins, mild losses |
| B — medium left tail | 0.76 | 1.326 | 0.55 | frequent small wins, medium losses |
| C — fat left tail | 0.88 | 1.255 | 0.30 | almost-always tiny wins, occasional severe loss |

**Three agents** with CRRA risk aversion $\gamma \in \{0.5, 1, 2\}$. Each agent considers all three models, picks an $f^\star$ under each, and selects among them by realized mean utility on a 1000-draw validation set from the true gamble.

**What we expect, and why.** CRRA utility at low $\gamma$ is dominated by the mean of the payoff. At high $\gamma$ it is dominated by the loss state, because the curvature of $u_\gamma$ grows with $\gamma$ and the loss state is where wealth gets hit hardest. So agents at different $\gamma$ weight different features of the same gamble differently.

Among our three candidates, A's thin-tailed worldview produces aggressive $f^\star$ recommendations and C's fat-tailed worldview produces conservative ones. This is so because each candidate's bias about the tail translates into a bias in $f^\star$ that happens to match the level of leverage an agent at a particular $\gamma$ would rationally choose under the truth. The low-$\gamma$ agent finds A's bias congenial. The high-$\gamma$ agent finds C's bias congenial. They are not picking the most accurate model; they are picking the model whose biases align with their own decision-loss profile.

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar

rng = np.random.default_rng(42)

p_true, Mp_true, Mm_true = 0.55, 1.5, 0.7
models = {
    "A (thin left tail)":   {"q": 0.31, "M_plus": 1.897, "M_minus": 0.80},
    "B (medium left tail)": {"q": 0.76, "M_plus": 1.326, "M_minus": 0.55},
    "C (fat left tail)":    {"q": 0.88, "M_plus": 1.255, "M_minus": 0.30},
}
gammas = [0.5, 1.0, 2.0]

# Verify all candidates agree with the truth on the mean
print(f"truth mean: {p_true*Mp_true + (1-p_true)*Mm_true:.4f}")
for name, m in models.items():
    print(f"{name}: mean = {m['q']*m['M_plus'] + (1-m['q'])*m['M_minus']:.4f}")

truth mean: 1.1400
A (thin left tail): mean = 1.1401
B (medium left tail): mean = 1.1398
C (fat left tail): mean = 1.1404


## Optimization stage: $f^\star$ under each candidate model

Each agent computes
$$
f^\star(\gamma, \text{model}) = \arg\max_f \; \mathbb{E}_{\text{model}}\bigl[u_\gamma(1 + f(M - 1))\bigr]
$$
with the expectation taken under the model's own beliefs $(q, M_+, M_-)$.

In [2]:
def neg_eu(f, q, Mp, Mm, gamma):
    wW = 1.0 + f * (Mp - 1.0)
    wL = 1.0 + f * (Mm - 1.0)
    if wW <= 0 or wL <= 0:
        return 1e10
    if gamma == 1:
        return -(q * np.log(wW) + (1 - q) * np.log(wL))
    return -(q * (wW ** (1 - gamma) - 1) / (1 - gamma)
             + (1 - q) * (wL ** (1 - gamma) - 1) / (1 - gamma))

def f_star(q, Mp, Mm, gamma, f_max=5.0):
    f_ruin = 1.0 / (1 - Mm) if Mm < 1 else f_max
    upper = min(f_ruin - 1e-4, f_max)
    res = minimize_scalar(neg_eu, args=(q, Mp, Mm, gamma),
                          bounds=(0.0, upper), method="bounded",
                          options={"xatol": 1e-7})
    return res.x

fstar_table = pd.DataFrame(
    {name: [f_star(m["q"], m["M_plus"], m["M_minus"], g) for g in gammas]
     for name, m in models.items()},
    index=pd.Index(gammas, name="gamma"),
)
fstar_table.round(3)

,A (thin left tail),B (medium left tail),C (fat left tail)
gamma,,,
0.5,1.791,1.582,1.169
1.0,0.781,0.953,0.787
2.0,0.355,0.511,0.453


Notice the pattern. As $\gamma$ rises (top to bottom), every model recommends less leverage. But the *cross-model* ordering is preserved: at every $\gamma$, A recommends the most aggressive $f^\star$ and C recommends the most conservative one. That is because A has the mildest perceived loss state and C has the worst — the thinner the perceived left tail, the more leverage the agent is willing to take.

## Validation stage: 1000 draws from the truth

Each $(\gamma, \text{model})$ pair's realized mean utility on 1000 i.i.d. realizations of the true gamble:
$$
\frac{1}{n} \sum_{t} u_\gamma\bigl(1 + f^\star_{\gamma, \text{model}}\,(M_t - 1)\bigr).
$$
The winning model per $\gamma$ is the one whose recommended $f^\star$ delivers the highest realized utility on data drawn from the true process.

In [3]:
n_draws = 1000
wins = rng.random(n_draws) < p_true
payoffs = np.where(wins, Mp_true, Mm_true)
print(f"realized win frequency: {wins.mean():.3f}  (truth p = {p_true})")

def realized_mean_utility(f, payoffs, gamma):
    w = 1.0 + f * (payoffs - 1.0)
    if np.any(w <= 0):
        return -np.inf
    if gamma == 1:
        return np.log(w).mean()
    return ((w ** (1 - gamma) - 1) / (1 - gamma)).mean()

val_table = pd.DataFrame(
    {name: [realized_mean_utility(fstar_table.loc[g, name], payoffs, g) for g in gammas]
     for name in models},
    index=pd.Index(gammas, name="gamma"),
)
winners = val_table.idxmax(axis=1).rename("winner")
pd.concat([val_table.round(5), winners], axis=1)

realized win frequency: 0.545  (truth p = 0.55)


,A (thin left tail),B (medium left tail),C (fat left tail),winner
gamma,,,,
0.5,0.11968,0.11834,0.10532,A (thin left tail)
1.0,0.05819,0.05916,0.05829,B (medium left tail)
2.0,0.02793,0.02853,0.02909,C (fat left tail)


## What we just saw

Each $\gamma$ picked a different model:

- **$\gamma = 0.5$ picks A (the thin-tail model).** A low-$\gamma$ agent's decision-loss profile rewards aggressive leverage and forgives loss-state outcomes. A's tail bias — claiming the loss state is milder than it really is — recommends exactly the kind of aggressive $f^\star$ the agent would choose under the truth. The other two candidates' worldviews recommend less leverage than the agent rationally wants given her $\gamma$.

- **$\gamma = 2$ picks C (the fat-tail model).** A high-$\gamma$ agent's decision-loss profile is dominated by the loss state. C's tail bias — claiming the loss state is more severe than it really is — recommends exactly the kind of conservative $f^\star$ the agent would choose under the truth. Note the irony: A's perceived $M_- = 0.80$ is closer to the truth's $0.70$ than C's $M_- = 0.30$, so by any reasonable measure of statistical accuracy on the tail, A is the more accurate model. Yet the high-$\gamma$ agent rationally prefers C, because C's *wrongness* is in the direction that produces the right $f^\star$ for her.

- **$\gamma = 1$ (log utility, the Kelly agent) picks B (the medium-tail model).** The Kelly agent is in the middle — and B's intermediate bias produces an intermediate $f^\star$ that lines up with what she would have chosen under the truth.

**The pedagogical punchline.** Each agent is picking the model whose biases happen to align with the decision losses she cares about most. Statistical accuracy is not the right model-selection criterion when a decision is downstream, and "pick the model that is most accurate on the feature you care about" is *also* not the right criterion — the high-$\gamma$ agent does not pick the model that is most accurate about the tail. The right criterion is realized decision payoff on data the model has not seen, and that criterion is risk-aversion-dependent.

**Robustness.** At $n = 1000$ there is real sampling noise in the realized win frequency $\hat p$, so on some seeds the winners shuffle around. Asymptotically (try $n = 100{,}000$) the canonical pattern A→B→C across the three $\gamma$ is the deterministic outcome.

**Connection to notebook 2.** Same principle as `06_Portfolio_optimization_and_copulas/2_simple_portfolio_optimization.ipynb`: there the candidates are continuous return distributions (Normal, Student-$t$, Johnson $S_U$), each biased about the tail in its own way, and the validation-stage winner depends on $\gamma$ for exactly this reason. CRPS would rank those candidates by bulk statistical accuracy; decision-payoff selection ranks them by which model's biases happen to produce useful $f^\star$ recommendations for the agent's risk aversion.